# 023 analysis

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.matching.benchmark import paired_frame_bootstrap
from uavloc.track import experiment_dir, load_runs

# Params

In [ ]:
EXP = "023-learned-correlation-head"
REFERENCE_EXP = "007-search-prior-and-gate"
REFERENCE_RUN = "007.1"
BASELINE_RUN = "023.1"
WEIGHTS_RUNS = ["023.2", "023.4"]
HEAD_RUNS = ["023.3", "023.5"]
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
BAR_DELTA_M = -0.5
GUARD_DELTA_M = 0.5
HARM_MAX = 0.084                  # 007.1's harm @ 10 m
REPRODUCE_MEAN_TOL_M = 0.05
REPRODUCE_DRAW_TOL_M = 0.01
REPRODUCE_DRAW_SHARE = 0.95
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42

# Data

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
runs = {r["run"]: r for r in load_runs(EXP)}
ARMS = {BASELINE_RUN: "ncc", **{r: "weights" for r in WEIGHTS_RUNS}, **{r: "head" for r in HEAD_RUNS}}
present = [r for r in ARMS if r in runs]
present

In [ ]:
def load_records(run: str, prefix: str = "") -> pd.DataFrame:
    return pd.read_csv(artifacts / f"{prefix}records-{run}.csv")

In [ ]:
records = {run: load_records(run) for run in present}
holdout = {run: load_records(run, "holdout_") for run in present}
reference = pd.read_csv(experiment_dir(REFERENCE_EXP) / "artifacts" / f"records-{REFERENCE_RUN}.csv")

# Reproduction

In [ ]:
key = ["frame", "noise_m", "bearing_deg"]
merged = records[BASELINE_RUN].round({"bearing_deg": 6}).merge(reference.round({"bearing_deg": 6}), on=key, suffixes=("", "_ref"))
at_primary = merged[merged["noise_m"] == PRIMARY_NOISE_M]
reproduction = {"draws_matched": len(merged), "draws": len(records[BASELINE_RUN]),
                "mean_diff_m_at_10": float(at_primary["error_m"].mean() - at_primary["error_m_ref"].mean()),
                "share_within_tol": float(((merged["error_m"] - merged["error_m_ref"]).abs() <= REPRODUCE_DRAW_TOL_M).mean())}
reproduction["pass"] = (abs(reproduction["mean_diff_m_at_10"]) <= REPRODUCE_MEAN_TOL_M
                        and reproduction["share_within_tol"] >= REPRODUCE_DRAW_SHARE)
reproduction

# Runs

In [ ]:
def run_row(run: str) -> dict:
    m = runs[run]["metrics"]
    return {"run": run, "arm": ARMS[run], "seed": runs[run].get("seed"),
            **{f"@{n:g}": m[f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
            "harm@10": m["harm_share_at_10m"], "reachable@10": m["mean_error_reachable_m_at_10m"],
            "covered@10": m["mean_position_error_m_covered"], "blind@10": m["mean_position_error_m_blind"],
            "p90@10": m["p90_position_error_m_at_10m"], "holdout@10": m["holdout_mean_position_error_m"],
            "holdout_harm@10": m["holdout_harm_share"], "ms": m["ms_per_correction"]}

In [ ]:
table = pd.DataFrame([run_row(r) for r in present]).set_index("run")
table.round(3)

# Paired

In [ ]:
def blind_frames(run_records: pd.DataFrame) -> set[str]:
    """Frames whose query holds no hand-weighted class: 023.1 logged coverage 0 on every draw."""
    ok = run_records[run_records["status"] == "ok"]
    return set(ok.groupby("frame")["coverage"].max().loc[lambda c: c == 0].index)

In [ ]:
def paired(a: pd.DataFrame, b: pd.DataFrame, noise_m: float, frames: set[str] | None = None) -> dict[str, float]:
    """`a - b` per frame at `noise_m`, frame-bootstrap CI; `frames` limits both to a subset."""
    if frames is not None:
        a, b = a[a["frame"].isin(frames)], b[b["frame"].isin(frames)]
    return paired_frame_bootstrap(a, b, noise_m, N_BOOTSTRAP, CI_PCT, SEED)

In [ ]:
blind = blind_frames(records[BASELINE_RUN])
covered = set(records[BASELINE_RUN]["frame"]) - blind
holdout_blind = blind_frames(holdout[BASELINE_RUN])
len(blind), len(covered), len(holdout_blind)

In [ ]:
rows = []
for run in [r for r in present if r != BASELINE_RUN]:
    for noise_m in NOISE_M:
        rows.append({"run": run, "bench": "primary", "subset": "all", "noise_m": noise_m,
                     **paired(records[run], records[BASELINE_RUN], noise_m)})
    for subset, frames in (("covered", covered), ("blind", blind)):
        rows.append({"run": run, "bench": "primary", "subset": subset, "noise_m": PRIMARY_NOISE_M,
                     **paired(records[run], records[BASELINE_RUN], PRIMARY_NOISE_M, frames)})
    rows.append({"run": run, "bench": "holdout", "subset": "all", "noise_m": PRIMARY_NOISE_M,
                 **paired(holdout[run], holdout[BASELINE_RUN], PRIMARY_NOISE_M)})
    rows.append({"run": run, "bench": "holdout", "subset": "covered", "noise_m": PRIMARY_NOISE_M,
                 **paired(holdout[run], holdout[BASELINE_RUN], PRIMARY_NOISE_M,
                          set(holdout[BASELINE_RUN]["frame"]) - holdout_blind)})
deltas = pd.DataFrame(rows)
deltas.round(3)

In [ ]:
def delta(run: str, bench: str, noise_m: float, subset: str = "all") -> pd.Series:
    return deltas[(deltas["run"] == run) & (deltas["bench"] == bench) & (deltas["noise_m"] == noise_m)
                  & (deltas["subset"] == subset)].iloc[0]

In [ ]:
def bar_check(run: str) -> dict[str, bool]:
    """The proposal's bar, item by item."""
    primary = delta(run, "primary", PRIMARY_NOISE_M)
    return {"delta_10": primary["diff_m"] <= BAR_DELTA_M and primary["ci_hi"] < 0,
            "guard_5": delta(run, "primary", 5.0)["ci_hi"] < GUARD_DELTA_M,
            "guard_15": delta(run, "primary", 15.0)["ci_hi"] < GUARD_DELTA_M,
            "harm_10": runs[run]["metrics"]["harm_share_at_10m"] <= HARM_MAX,
            "holdout_not_worse": delta(run, "holdout", PRIMARY_NOISE_M)["diff_m"] < 0}

In [ ]:
bars = pd.DataFrame({run: bar_check(run) for run in present if run != BASELINE_RUN}).T
bars["all"] = bars.all(1)
bars

In [ ]:
verdict = all(bars.loc[r, "all"] for r in HEAD_RUNS if r in bars.index) and all(r in bars.index for r in HEAD_RUNS)
verdict

In [ ]:
head_vs_weights = pd.DataFrame([{"seed_runs": f"{h} - {w}", **paired(records[h], records[w], PRIMARY_NOISE_M)}
                                for h, w in zip(HEAD_RUNS, WEIGHTS_RUNS) if h in records and w in records])
head_vs_weights.round(3)

# Training

In [ ]:
def final_gap(run: str) -> dict[str, float]:
    last = pd.read_csv(artifacts / f"history-{run}.csv").iloc[-1]
    return {"run": run, "train_error_m": last["train_error_m"], "val_error_m": last["val_error_m"],
            "val_over_train": last["val_error_m"] / last["train_error_m"], "train_loss": last["train_loss"],
            "val_loss": last["val_loss"], "val_within_1m_share": last["val_within_1m_share"]}

In [ ]:
gaps = pd.DataFrame([final_gap(r) for r in WEIGHTS_RUNS + HEAD_RUNS if r in runs]).set_index("run")
gaps.round(3)

In [ ]:
learned_weights = {}
for run in WEIGHTS_RUNS:
    if run in runs:
        state = torch.load(artifacts / f"head-{run}.pt", map_location="cpu")
        learned_weights[run] = {"weights": state["weights"].numpy().round(3).tolist(),
                                "temperature": float(state["log_temperature"].exp())}
learned_weights

# Plots

In [ ]:
def plot_cdfs(by_run: dict[str, pd.DataFrame], noise_m: float, title: str) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6, 4))
    for run, r in by_run.items():
        errors = np.sort(r.loc[r["noise_m"] == noise_m, "error_m"].to_numpy())
        ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{run} {ARMS[run]}")
    ax.axvline(noise_m, color="k", ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws"); ax.set_title(title)
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    return fig

In [ ]:
def primary_frame_means(run_records: pd.DataFrame) -> pd.Series:
    return run_records[run_records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean()

In [ ]:
def plot_frame_deltas(run: str, by_run: dict[str, pd.DataFrame], title: str) -> plt.Figure:
    diff = (primary_frame_means(by_run[run]) - primary_frame_means(by_run[BASELINE_RUN])).sort_values()
    fig, ax = plt.subplots(figsize=(12, 3.5))
    diff.plot.bar(ax=ax, width=.8, color=["tab:green" if d < 0 else "tab:red" for d in diff])
    ax.axhline(0, color="k", lw=.8); ax.set_ylabel(f"{run} - {BASELINE_RUN}, m @ {PRIMARY_NOISE_M:g}"); ax.set_title(title)
    fig.tight_layout()
    return fig

In [ ]:
figures = {"cdf-primary": plot_cdfs(records, PRIMARY_NOISE_M, "primary bench @ 10 m"),
           "cdf-holdout": plot_cdfs(holdout, PRIMARY_NOISE_M, "holdout bench @ 10 m")}
figures.update({f"frames-{run}": plot_frame_deltas(run, records, f"per frame, {ARMS[run]}")
                for run in HEAD_RUNS + WEIGHTS_RUNS if run in records})

# Save

In [ ]:
figures_dir = experiment_dir(EXP) / "figures"
figures_dir.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(figures_dir / f"{name}.png", dpi=110)
deltas.to_csv(artifacts / "paired-deltas.csv", index=False)
(artifacts / "analysis.json").write_text(json.dumps(
    {"reproduction": reproduction, "bars": bars.astype(bool).to_dict(orient="index"), "verdict": bool(verdict),
     "learned_weights": learned_weights, "blind_frames": sorted(blind)}, indent=1, default=float))
verdict